# 2. 新旧读出捕获与批处理对照（GPU）

默认 Gemma-12B；完成后保存结果，开全新的 GPU 运行时，将 MODEL_KEY 改为 qwen9b 再运行一次。

[实验设置与运行说明](../../../docs/experiment_notes/03_section44_readout_correction/cf62bb7c6bbb369f.md)


In [1]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready.')

IMPLEMENTATION_SHA256 = '160372af558929c42134517d3ffc09527278d018135dd39a46f32eb6987dde7d'
IMPLEMENTATION_SHA256 = '3c9c616671c4ed6ec1ea9b9032a18d3bad8d742069ab38fbf290c722cc968071'


Initialize progress display:   0%|          | 0/1 [00:00<?, ?cell/s]

Cell progress helper ready.


In [2]:
%%cell_progress 1/15 Mount Drive
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


Mount Drive:   0%|          | 0/15 [00:00<?, ?cell/s]

Mounted at /content/drive
Project: /content/drive/MyDrive/VisualTorque


In [3]:
%%cell_progress 2/15 Install inference environment
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


Install inference environment:   7%|6         | 1/15 [00:00<?, ?cell/s]

torch: 2.11.0+cu128 | CUDA: 12.8
torchvision: 0.26.0+cu128
Toolkit source: /content/drive/MyDrive/VisualTorque/toolkit/src/visual_torque_toolkit/__init__.py


In [4]:
%%cell_progress 3/15 Check GPU access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select a GPU runtime with enough memory for the pinned model.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


Check GPU access:  13%|#3        | 2/15 [00:00<?, ?cell/s]

NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA memory (GB): 95.0
Hugging Face login skipped: Secret HF_TOKEN does not exist.


In [5]:
%%cell_progress 4/15 Load shared validation utilities
"""Pure CPU utilities embedded in the follow-up notebooks; no model execution."""
import hashlib
import json
from pathlib import Path
from collections import defaultdict, Counter
from itertools import combinations, product
import numpy as np
import pandas as pd


def plain(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=plain, allow_nan=False).encode()).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def read_jsonl(path):
    return [json.loads(s) for s in Path(path).read_text(encoding='utf-8').splitlines() if s.strip()]


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=plain, allow_nan=False) + '\n', encoding='utf-8')
    temp.replace(path)


def freeze(path, identity):
    identity = json.loads(json.dumps(identity, default=plain, sort_keys=True, allow_nan=False))
    if Path(path).is_file():
        assert read_json(path) == identity, f'Configuration changed; preserve the old run and use a new RUN_ID: {path}'
    else:
        save_json(path, identity)
    return digest(identity)


def cached_json(directory, key, identity, compute):
    path = Path(directory) / (digest(str(key))[:24] + '.json')
    expected = digest(identity)
    if path.is_file():
        old = read_json(path)
        assert old['key'] == str(key) and old['identity'] == expected, f'Incompatible cache: {path}'
        return old['result']
    value = compute()
    save_json(path, dict(key=str(key), identity=expected, result=value))
    return value


def bootstrap(values, key='', replicates=10000):
    x = np.asarray(values, dtype=float)
    assert x.ndim == 1 and len(x) and np.isfinite(x).all()
    rng = np.random.default_rng(int(digest(str(key))[:16], 16))
    draws = np.concatenate([rng.choice(x, (min(250, replicates-i), len(x)), replace=True).mean(1)
                            for i in range(0, replicates, 250)])
    return dict(n=len(x), mean=float(x.mean()), lower=float(np.quantile(draws, .025)),
                upper=float(np.quantile(draws, .975)),
                p_value=float((1 + (draws <= 0).sum()) / (replicates + 1)))


def bh(values):
    p = np.asarray(values, dtype=float)
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
    order = np.argsort(p)
    q = np.empty_like(p)
    previous = 1.
    for rank in range(len(p)-1, -1, -1):
        i = order[rank]
        previous = min(previous, p[i]*len(p)/(rank+1))
        q[i] = previous
    return q


def paired(left, right, id_col, value):
    assert left[id_col].is_unique and right[id_col].is_unique
    assert set(left[id_col]) == set(right[id_col]), 'Different paired support'
    out = left[[id_col, value]].merge(right[[id_col, value]], on=id_col, validate='one_to_one', suffixes=('_l', '_r'))
    return out[value+'_l'].to_numpy(float) - out[value+'_r'].to_numpy(float)


def semantic_scores(record):
    if 'semantic_logprobs' in record:
        return {str(k): float(v) for k, v in record['semantic_logprobs'].items()}
    values = record['candidate_logprobs']
    if isinstance(values, dict):
        return {str(record['label_to_semantic'][k]): float(v) for k, v in values.items()}
    return dict(zip(map(str, record['candidate_semantics']), map(float, values)))


def correct_margin(record):
    values = semantic_scores(record)
    gold = str(record['gold_semantic'])
    assert gold in values and len(values) >= 2
    return values[gold] - max(v for k, v in values.items() if k != gold)


def random_unit(shape, key):
    r = np.random.default_rng(int(digest(key)[:16], 16)).normal(size=shape)
    return r / np.linalg.norm(r)


def coordinate_clamp(pre, untreated, axis, cast):
    h, base, u = [np.asarray(x, dtype=np.float64) for x in (pre, untreated, axis)]
    u = u / np.linalg.norm(u)
    a = float(np.sum((h-base)*u))
    return np.asarray(cast(h-a*u), dtype=np.float64), a


def native_norm_edit(pre, direction, norm, cast, tolerance=.02, steps=48):
    h = np.asarray(pre, dtype=np.float64)
    r = np.asarray(direction, dtype=np.float64)
    assert np.isfinite(h).all() and np.isfinite(r).all() and norm >= 0 and np.linalg.norm(r) > 0
    r /= np.linalg.norm(r)
    if norm == 0:
        return h.copy(), dict(actual_norm=0., target_norm=0., relative_error=0., matched=True)
    lo, hi = 0., max(float(norm), 1e-12)
    for _ in range(32):
        if np.linalg.norm(np.asarray(cast(h+hi*r))-h) >= norm:
            break
        hi *= 2
    best, error = h.copy(), 1.
    for _ in range(steps):
        mid = (lo+hi)/2
        candidate = np.asarray(cast(h+mid*r), dtype=np.float64)
        actual = float(np.linalg.norm(candidate-h))
        current = abs(actual-norm)/norm
        if current < error:
            best, error = candidate.copy(), current
        if error <= tolerance:
            break
        if actual < norm:
            lo = mid
        else:
            hi = mid
    return best, dict(actual_norm=float(np.linalg.norm(best-h)), target_norm=float(norm),
                      relative_error=float(error), matched=bool(error <= tolerance))


def equal_removal(pre, untreated, axis, random_direction, cast, tolerance=.02):
    target, coefficient = coordinate_clamp(pre, untreated, axis, cast)
    target_norm = float(np.linalg.norm(target-np.asarray(pre)))
    sign = 1. if coefficient >= 0 else -1.
    post, audit = native_norm_edit(pre, -sign*np.asarray(random_direction), target_norm, cast, tolerance)
    audit.update(reference_coefficient=coefficient, reference_actual_norm=target_norm)
    return post, audit


def geometry_signature(stacks):
    a = tuple(sorted((str(s['side']), int(s['distance']), int(s['units'])) for s in stacks))
    b = tuple(sorted(('right' if side == 'left' else 'left', d, u) for side, d, u in a))
    return min(a, b)  # mirrors belong to the same physical family


def geometries_in(value):
    found = set()
    if isinstance(value, dict):
        stacks = value.get('stacks')
        if isinstance(stacks, list) and stacks and all(isinstance(s, dict) and {'side','distance','units'} <= set(s) for s in stacks):
            found.add(geometry_signature(stacks))
        # Some prior matched cohorts store geometry instead of stack records.
        geom = value.get('pair_geometry')
        if isinstance(geom, (list, tuple)) and len(geom) == 4 and all(isinstance(x, (int,float)) for x in geom):
            u1,d1,u2,d2 = map(int, geom)
            found.add(geometry_signature([dict(side='left',distance=d1,units=u1),dict(side='right',distance=d2,units=u2)]))
        if isinstance(geom, (list, tuple)) and len(geom)==2 and all(isinstance(x,(list,tuple)) and len(x)==2 for x in geom):
            (u1,d1),(u2,d2)=geom
            found.add(geometry_signature([dict(side='left',distance=int(d1),units=int(u1)),dict(side='right',distance=int(d2),units=int(u2))]))
        for child in value.values():
            found.update(geometries_in(child))
    elif isinstance(value, list):
        for child in value:
            found.update(geometries_in(child))
    return found


def physics(stacks):
    units = {s:sum(int(x['units']) for x in stacks if x['side']==s) for s in ('left','right')}
    dist = {s:sum(int(x['distance']) for x in stacks if x['side']==s) for s in ('left','right')}
    torque = {s:sum(int(x['units'])*int(x['distance']) for x in stacks if x['side']==s) for s in ('left','right')}
    out = lambda x:'left' if x>0 else 'right' if x<0 else 'balanced'
    net=torque['left']-torque['right']; w=units['left']-units['right']; d=dist['left']-dist['right']
    return dict(net_torque=net,outcome=out(net),weight_score=w,distance_score=d,
                normalized_margin=abs(net)/max(1,sum(torque.values())),
                left_units=units['left'],right_units=units['right'],left_torque=torque['left'],right_torque=torque['right'],
                strategy_predictions=dict(exact_torque=out(net),weight_only=out(w),distance_only=out(d)))


def exclusion_snapshot(project, followup_root, dataset_dir=None):
    lock=Path(followup_root)/'prior_geometry_snapshot.json'
    if lock.is_file():
        old=read_json(lock)
        for path, sha in old['files'].items():
            assert file_hash(Path(path))==sha, f'Prior input changed: {path}'
        return {tuple(tuple(x) for x in sig) for sig in old['signatures']}, old
    paths={Path(dataset_dir or project/'data/dataset_v1')/'scenes.jsonl'}
    for pattern in ('*manifest*.jsonl','*support*.jsonl','*geometries*.jsonl'):
        paths.update(p for p in (project/'results').rglob(pattern) if Path(followup_root) not in p.parents)
    excluded=set(); hashes={}
    for path in sorted(paths):
        assert path.is_file(), f'Missing required geometry input: {path}'
        excluded.update(geometries_in(read_jsonl(path)))
        hashes[str(path.resolve())]=file_hash(path)
    assert excluded, 'No prior physical geometry was indexed'
    old=dict(files=hashes,signatures=sorted(excluded),mirror_canonical=True)
    save_json(lock,old)
    return excluded,old


def inverse_families(n, seed, excluded):
    assert n%4==0
    pool=defaultdict(list)
    for dl in combinations(range(1,7),2):
        for dr in combinations(range(1,7),2):
            for units in product(range(1,6), repeat=4):
                stacks=[dict(side=side,distance=d,units=u) for side,d,u in zip(('left','left','right','right'),dl+dr,units)]
                f=physics(stacks)
                pool[dl+dr].append((stacks,f))
    candidates=[]
    for distances,rows in pool.items():
        by_net=defaultdict(list)
        for stacks,f in rows:
            if f['outcome']==f['strategy_predictions']['weight_only'] and f['outcome']!='balanced':
                by_net[f['net_torque']].append((stacks,f))
        for stacks,f in rows:
            margin=f['normalized_margin']; difficulty='hard' if .08<=margin<.22 else 'moderate' if .22<=margin<.45 else None
            if (difficulty and f['outcome']!='balanced' and f['strategy_predictions']['weight_only'] not in ('balanced',f['outcome'])
                    and f['strategy_predictions']['distance_only']==f['outcome'] and abs(f['distance_score'])>=2):
                sources=by_net.get(f['net_torque'],[])
                if sources:candidates.append((digest([seed,stacks]),difficulty,stacks,f,sources))
    candidates.sort(key=lambda r:r[0]); used=set(excluded); counts=Counter(); result=[]
    for _,difficulty,stacks,f,sources in candidates:
        stratum=(difficulty,f['outcome'])
        if counts[stratum]>=n//4 or geometry_signature(stacks) in used:continue
        valid=[(s,g) for s,g in sources if geometry_signature(s) not in used|{geometry_signature(stacks)}]
        if not valid:continue
        valid.sort(key=lambda x:(sum(abs(a['units']-b['units']) for a,b in zip(stacks,x[0])),digest([seed,x[0]])))
        src,sf=valid[0];fid='fixed_'+digest([seed,stacks,src])[:20]
        roles={name:dict(scene_id=fid+'__'+name,image_filename=fid+'__'+name+'.png',stacks=s,features=g)
               for name,s,g in [('base',stacks,f),('aligned_source',src,sf)]}
        result.append(dict(family_id=fid,remapping_split='new_fixed_confirmation',roles=roles,margin_bin=difficulty))
        used.update([geometry_signature(stacks),geometry_signature(src)]);counts[stratum]+=1
        if len(result)==n:break
    assert len(result)==n and set(counts.values())=={n//4}, (len(result),counts)
    return sorted(result,key=lambda r:r['family_id'])


def repair_families(n, seed, excluded):
    assert n%4==0
    candidates=defaultdict(list)
    for ul,dl,ur,dr in product(range(1,9),range(1,7),range(1,9),range(1,7)):
        s=[dict(side='left',units=ul,distance=dl),dict(side='right',units=ur,distance=dr)]
        f=physics(s)
        if f['net_torque']==0:continue
        if ul==ur and dl!=dr:kind='equal_weight_distance'
        elif f['strategy_predictions']['weight_only'] not in ('balanced',f['outcome']):kind='weight_distance_conflict'
        elif f['strategy_predictions']['weight_only']==f['outcome'] and f['strategy_predictions']['distance_only']==f['outcome']:kind='aligned_cues'
        else:continue
        candidates[kind].append(s)
    for units in product(range(1,6),repeat=4):
        for distances in [((1,3),(2,5)),((2,5),(1,4)),((1,6),(3,5)),((2,4),(3,6))]:
            s=[dict(side=side,distance=d,units=u) for side,d,u in zip(('left','left','right','right'),distances[0]+distances[1],units)]
            if physics(s)['net_torque']!=0:candidates['multi_stack_sum'].append(s)
    used=set(excluded); result=[]
    for kind in ('equal_weight_distance','weight_distance_conflict','multi_stack_sum','aligned_cues'):
        rows=sorted(candidates[kind],key=lambda s:digest([seed,kind,s])); count=0
        for stacks in rows:
            sig=geometry_signature(stacks)
            if sig in used:continue
            fid='repair_'+digest([seed,kind,sig])[:20];scenes=[]
            for mirror in (False,True):
                current=[dict(s,side=('right' if s['side']=='left' else 'left')) if mirror else dict(s) for s in stacks]
                for style in ('standard','alternate'):
                    sid=f'{fid}__m{int(mirror)}__{style}'
                    scenes.append(dict(scene_id=sid,image_filename=sid+'.png',stacks=current,features=physics(current),style=style,mirror=mirror))
            result.append(dict(family_id=fid,stratum=kind,scenes=scenes));used.add(sig);count+=1
            if count==n//4:break
        assert count==n//4,(kind,count,'Insufficient unused geometries; do not silently change task ranges')
    return sorted(result,key=lambda r:r['family_id'])


def csv_markdown(frame, columns=None, limit=40):
    if frame.empty:
        return '无记录。'
    shown = frame if columns is None else frame[columns]
    shown = shown.head(limit)
    escape = lambda x: str(x).replace('|', '/').replace('\n', ' ')
    lines = ['| '+' | '.join(map(escape, shown.columns))+' |', '| '+' | '.join(['---']*len(shown.columns))+' |']
    for row in shown.itertuples(index=False, name=None):
        lines.append('| '+' | '.join(escape(f'{v:.6g}' if isinstance(v, float) else v) for v in row)+' |')
    if len(frame) > limit:
        lines += ['', f'此处显示前 {limit} 行，完整 {len(frame)} 行保存在 CSV。']
    return '\n'.join(lines)

"""Small registered-output packet; never include stale files from another run."""


class DiagnosticPacket:
    def __init__(self, root, identity):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        freeze(self.root/'protocol.json', identity)
        self.outputs = {self.root/'protocol.json'}
        self.inputs = {}

    def track(self, path):
        path = Path(path)
        self.inputs[str(path)] = file_hash(path)
        return path

    def csv(self, name, frame):
        path = self.root/name
        frame.to_csv(path, index=False, encoding='utf-8-sig')
        self.outputs.add(path)
        return path

    def json(self, name, value):
        path = self.root/name
        save_json(path, value)
        self.outputs.add(path)
        return path

    def text(self, name, value):
        path = self.root/name
        path.write_text(value, encoding='utf-8')
        self.outputs.add(path)
        return path

    def finish(self, summary, report, archive_name, auto_download=True):
        import zipfile
        changed = [p for p, h in self.inputs.items() if not Path(p).is_file() or file_hash(p) != h]
        summary = dict(summary, source_files_unchanged=not changed, changed_inputs=changed)
        if changed:
            summary['status'] = 'inputs_changed_results_not_comparable'
        self.json('input_manifest.json', self.inputs)
        self.json('RUN_COMPLETE.json', summary)
        self.text('REPORT_ZH.md', report)
        archive = self.root/archive_name
        temporary = archive.with_suffix('.zip.tmp')
        with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
            for path in sorted(self.outputs):
                bundle.write(path, str(path.relative_to(self.root)))
        temporary.replace(archive)
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        print('完整结果包：', archive)
        if auto_download:
            try:
                from google.colab import files
                files.download(str(archive))
            except Exception as error:
                print('自动下载未触发，请从以上 Drive 路径手动下载：', type(error).__name__, str(error))
        return archive


def diagnostic_show(title, frame, limit=30):
    print('\n'+title)
    if frame.empty:
        print('无记录或此项输入缺失；请查看 coverage。')
        return
    with pd.option_context('display.max_rows', limit+2, 'display.max_columns', None, 'display.max_colwidth', 90):
        display(frame.head(limit))
    if len(frame) > limit:
        print(f'这里显示前 {limit} 行；结果 ZIP 保存全部 {len(frame)} 行。')


"""Pure pairing/scheduling and readout comparisons for a bounded capture bridge.

This reconstructs the checked local discovery template in the current runtime.
It never claims to restore an unlogged historical environment or batch schedule.
"""


def capture_key(record):
    return str(record['scene_id']), str(record['variant'])


def candidate_batch_schedule(ordered_keys, wanted, batch_size):
    assert isinstance(batch_size, int) and 1 < batch_size <= 8
    assert len(ordered_keys) == len(set(ordered_keys)) and set(wanted) <= set(ordered_keys)
    batches = [ordered_keys[i:i+batch_size] for i in range(0, len(ordered_keys), batch_size)]
    selected = [b for b in batches if set(b) & set(wanted)]
    covered = [k for b in selected for k in b if k in wanted]
    assert len(covered) == len(wanted) and set(covered) == set(wanted)
    return selected


def normalize_sha(value):
    return str(value).removeprefix('sha256:')


def exact_input_comparison(left, right):
    """Fingerprints include names, dtypes, shapes and full tensor bytes."""
    keys = sorted(set(left) | set(right))
    return [dict(field=k, equal=k in left and k in right and left[k] == right[k],
                 legacy=left.get(k), current=right.get(k)) for k in keys]


def aggregate_capture_scenes(keys, arrays, metadata, old_means, threshold, decode_fn):
    assert len(keys) == len(set(keys)) and len(metadata) == len(keys)
    assert all(len(a) == len(keys) for a in arrays.values())
    index = pd.DataFrame([dict(scene_id=s, variant=v, index=i) for i, (s, v) in enumerate(keys)])
    assert index.groupby('scene_id').size().eq(6).all()
    assert index.groupby('scene_id').variant.nunique().eq(6).all()
    out = []
    classify = lambda x: 'left' if x > threshold else 'right' if x < -threshold else 'balanced'
    for sid, group in index.groupby('scene_id', sort=True):
        inds = group['index'].to_numpy(int)
        row = dict(scene_id=sid, mappings=6)
        if sid in old_means:
            row['historical_readout'] = decode_fn(old_means[sid])
            row['historical_class'] = classify(row['historical_readout'])
        for name, array in arrays.items():
            # All variants are first represented in float32. Legacy storage then
            # rounds the six-map mean to FP16, exactly as the local template.
            mean = np.asarray(array[inds], np.float32).mean(0)
            if name.endswith('_fp16'):
                mean = mean.astype(np.float16).astype(np.float32)
            value = decode_fn(mean)
            row[name+'_readout'] = value
            row[name+'_class'] = classify(value)
            if sid in old_means:
                row[name+'_archive_gap'] = value-row['historical_readout']
                row[name+'_archive_class_changed'] = classify(value) != row['historical_class']
        out.append(row)
    return pd.DataFrame(out)


def scene_gap_summary(scenes, route_names):
    rows = []
    for name in route_names:
        key = name+'_archive_gap'
        if key not in scenes:
            continue
        gap = scenes[key].dropna()
        rows.append(dict(route=name, scenes=len(gap), mean_absolute_archive_gap=float(gap.abs().mean()),
            maximum_absolute_archive_gap=float(gap.abs().max()), mean_signed_archive_gap=float(gap.mean()),
            class_changed_scenes=int(scenes.loc[gap.index, name+'_archive_class_changed'].sum())))
    return pd.DataFrame(rows)


def bridge_classification(preparation_equal, storage_parity, original_mean_consistent, summaries, single_batch_gap):
    notes = []
    if not preparation_equal:
        notes.append('旧模板与当前单张流程的实际输入或候选计划不同；先定位具体输入字段，批量比较不能解释全部差异。')
    if not storage_parity:
        notes.append('同一次前向的旧式 FP16 捕获与原生状态转存不一致；检查捕获位置或输出处理。')
    if original_mean_consistent is False:
        notes.append('历史逐映射分片重建出的均值与历史 scene_mean 文件不一致；历史缓存内部存在待核对差异。')
    elif original_mean_consistent is None:
        notes.append('缺少完整历史逐映射分片；无法核对历史均值的来源或原批次成员。')
    if single_batch_gap is not None:
        notes.append('同一当前环境中，所选批处理上下文与单张处理的差异已测量；它同时包含批处理、填充及可能的图像处理差异。')
    names = set(summaries.route) if not summaries.empty else set()
    if {'legacy_single_fp16','legacy_batch_fp16'} <= names:
        a = summaries.set_index('route').loc['legacy_single_fp16', 'mean_absolute_archive_gap']
        b = summaries.set_index('route').loc['legacy_batch_fp16', 'mean_absolute_archive_gap']
        notes.append('批处理路线与历史均值更接近。' if b < a else '本次候选批处理路线没有降低平均历史读出差异。')
    notes.append('较接近历史缓存不等于已恢复历史流程；未记录的批次组成、软件或硬件仍不能自动归因。此诊断不更新任何科学确认状态。')
    return notes


Load shared validation utilities:  20%|##        | 3/15 [00:00<?, ?cell/s]

In [6]:
%%cell_progress 5/15 Prepare existing Dataset v1
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


Prepare existing Dataset v1:  27%|##6       | 4/15 [00:00<?, ?cell/s]

Extracting Dataset v1:   0%|          | 0/2106 [00:00<?, ?file/s]

Dataset: /content/visualtorque_dataset_v1


In [7]:
%%cell_progress 6/15 Select Gemma or Qwen-9B and preserve all previous experiments
MODEL_KEY='qwen9b'  # @param ["gemma12b", "qwen9b"]
# 换模型必须使用全新的 GPU 运行时。本份不运行 Qwen-27B。
RUN_ID='input_mapping_fix'  # 独立保存本次修正版；保留 main 中的未完成诊断包。
REPEAT_RUN_ID='main'
AUTO_DOWNLOAD=True
BATCH_SIZE_OVERRIDE=None  # 仅在掌握历史配置时填写；默认按旧模板候选（Gemma=2，Qwen=4）。
SPECS={'gemma12b':dict(model_id='google/gemma-4-12B-it',slug='google__gemma-4-12B-it',layers=48,
    revision='707f0a3b8a3c7ad586ed01e27eafbad8a27dd0f7',template_batch=2),
    'qwen9b':dict(model_id='Qwen/Qwen3.5-9B',slug='Qwen__Qwen3.5-9B',layers=32,
    revision='c202236235762e1c871ad0ccb60c8ee5ba337b9a',template_batch=4)}
assert MODEL_KEY in SPECS
SPEC=SPECS[MODEL_KEY];MODEL_ID=SPEC['model_id'];MODEL_REVISION=SPEC['revision'];MODEL_SLUG=SPEC['slug']
MODEL_DTYPE='bfloat16';DEVICE_MAP='auto';EXPECTED_LAYER_COUNT=SPEC['layers']
OUTPUT_ROOT=PROJECT_ROOT/'results/cross_model_independent_confirmation/capture_bridge'/RUN_ID/MODEL_SLUG
packet=DiagnosticPacket(OUTPUT_ROOT,dict(schema='readout-capture-bridge-1.0',implementation=IMPLEMENTATION_SHA256,
    model=MODEL_KEY,revision=MODEL_REVISION,batch_override=BATCH_SIZE_OVERRIDE,template_batch=SPEC['template_batch'],
    cohort='original 19 development scenes; six mappings; no refitting',source_repeat_run=REPEAT_RUN_ID,
    route='current single, local discovery-template single, candidate discovery-template batch in one current runtime',
    precision='BF16 forward; native float32 copies and actual FP16 observer',new_scientific_confirmation=False))
ready=True;issues=[];coverage=[];single_results={};single_arrays={};batch_results=[];batch_arrays={}
historical_states={};historical_records={};historical_means={};archive_order=[]
archive_mean_checks=[];source_original_mean_consistent=None;MODEL_FORWARD_ATTEMPTS=0;MODEL_EXAMPLES_ATTEMPTED=0

def bridge_issue(stage,error):
    global ready
    ready=False
    row=dict(stage=stage,status='incomplete',reason=type(error).__name__+': '+str(error))
    issues.append(row);print('此部分未完成，继续运行后面的打包单元格：',row)

print('模型：',MODEL_ID,'；输出：',OUTPUT_ROOT)


Select Gemma or Qwen-9B and preserve all previous experiments:  33%|###3      | 5/15 [00:00<?, ?cell/s]

模型： Qwen/Qwen3.5-9B ；输出： /content/drive/MyDrive/VisualTorque/results/cross_model_independent_confirmation/capture_bridge/input_mapping_fix/Qwen__Qwen3.5-9B


In [8]:
%%cell_progress 7/15 Define the two read-only capture conventions
def output_hidden_tensor(output):
    if torch.is_tensor(output):
        tensor = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        tensor = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        tensor = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported decoder-layer output type: {type(output).__name__}')
    if tensor.ndim != 3:
        raise ValueError(f'Expected [batch, sequence, hidden], got {tuple(tensor.shape)}')
    return tensor

def output_hidden(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and torch.is_tensor(output[0]):
        hidden = output[0]
    else:
        raise TypeError(f'Unsupported decoder output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def readout_value(hidden, mean, scale, coef, intercept):
    return float(((np.asarray(hidden, dtype=np.float64) - mean) / scale) @ coef + intercept)

def decode(hidden):
    return readout_value(hidden, PROBE_MEAN, PROBE_SCALE, PROBE_COEF, PROBE_INTERCEPT)

def probe_outcome(value):
    return 'left' if value > BALANCE_THRESHOLD else 'right' if value < -BALANCE_THRESHOLD else 'balanced'

CAPTURE_SOURCE_PROVENANCE={'legacy_template': 'VisualTorque_Activation_Discovery_A100.ipynb', 'legacy_capture_function_sha256': '4fa8939508126dd4cd5036ad4b63c338ab0fcae0f9d6a721041768ab5b28db33', 'current_template': 'VisualTorque_CrossModel_Readout_Repeatability_A100.ipynb', 'current_capture_function_sha256': 'f416f39995ed55a9208b836ed850eb15510cdf2efa26022777d8e62c5acab853', 'legacy_default_batches': {'qwen9b': 4, 'gemma12b': 2}, 'status': 'Checked local source pattern replayed in current runtime; historical implementation identity is not fully logged'}

"""No-edit model execution; embedded after Torch/model setup.

Old-style preparation and FP16 hook follow the checked discovery template;
current-style preparation/native hook follow the checked E1 notebook.
"""

from collections.abc import Mapping


def tensor_signature(value):
    if torch.is_tensor(value):
        v = value.detach().contiguous().cpu()
        raw = v.reshape(-1).view(torch.uint8).numpy().tobytes()
        return dict(dtype=str(v.dtype), shape=list(v.shape), sha256=hashlib.sha256(raw).hexdigest())
    if isinstance(value, (tuple, list)):
        return [tensor_signature(v) for v in value]
    # HF BatchFeature/BatchEncoding are mapping wrappers, not dict subclasses.
    # Inspect every contained value without changing the prepared model inputs.
    if isinstance(value, Mapping):
        return {str(k): tensor_signature(v) for k, v in sorted(value.items())}
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    raise TypeError('Untracked prepared-input value: '+type(value).__name__)


def legacy_prepare(examples):
    assert examples and all(e.labels == examples[0].labels for e in examples)
    plan = adapter._candidate_plan(examples[0])
    assert plan is not None
    prefix, tokens = plan
    assert all(adapter._candidate_plan(e) == plan for e in examples)
    conversations = [e.messages() for e in examples]
    prepared = adapter._append_text_prefix(
        adapter._to_device(adapter._template(conversations, add_generation_prompt=True)), prefix)
    return prepared, dict(prefix=str(prefix), token_by_label={str(k): int(v) for k, v in tokens.items()})


def current_prepare(example):
    plan = adapter._candidate_plan(example)
    assert plan is not None
    prefix, tokens = plan
    prepared = adapter._append_text_prefix(adapter._to_device(adapter._template(
        [example.messages()], add_generation_prompt=True)), prefix)
    return prepared, dict(prefix=str(prefix), token_by_label={str(k): int(v) for k, v in tokens.items()})


def prepared_rows(inputs, examples):
    """Record unpadded text and model-supported per-image fingerprints.

    The full tensor dictionary is always fingerprinted separately. A missing
    per-image partition is explicitly unknown, never assumed equivalent.
    """
    mask = inputs['attention_mask'].detach().cpu()
    ids = inputs['input_ids'].detach().cpu()
    assert ids.ndim == mask.ndim == 2 and ids.shape == mask.shape and len(ids) == len(examples)
    pixels = inputs.get('pixel_values')
    grid = inputs.get('image_grid_thw')
    pixel_rows = [None for _ in examples]
    method = 'per_image_partition_unavailable'
    if torch.is_tensor(pixels) and torch.is_tensor(grid) and len(grid) == len(examples):
        sizes = grid.detach().cpu().to(torch.int64).prod(-1).tolist()
        if sum(sizes) == pixels.shape[0]:
            start = 0
            for i, size in enumerate(sizes):
                pixel_rows[i] = tensor_signature(pixels[start:start+size])
                start += size
            method = 'image_grid_patch_partition'
    elif torch.is_tensor(pixels) and len(pixels) == len(examples):
        pixel_rows = [tensor_signature(pixels[i:i+1]) for i in range(len(examples))]
        method = 'leading_batch_dimension'
    out = []
    for i, ex in enumerate(examples):
        active = torch.nonzero(mask[i], as_tuple=False).flatten()
        assert len(active)
        out.append(dict(scene_id=str(ex.scene_id), variant=str(ex.variant), example_id=str(ex.example_id),
            unpadded_input_ids=ids[i][mask[i].bool()].tolist(), sequence_length=int(ids.shape[1]),
            active_token_count=len(active), last_active_index=int(active[-1]),
            selected_index=int(ids.shape[1]-1), selected_token_active=bool(mask[i,-1]),
            selected_token_id=int(ids[i,-1]), image_fingerprint=pixel_rows[i], image_partition=method))
    return out


def execute_bridge_forward(inputs, examples, plan):
    global MODEL_FORWARD_ATTEMPTS, MODEL_EXAMPLES_ATTEMPTED
    # Separate observers preserve both original storage/capture conventions.
    captured = {}
    handles = []
    def legacy_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        captured['fp16'] = hidden[:, -1, :].detach().to(device='cpu', dtype=torch.float16).float().numpy().copy()
    def native_hook(_module, _inputs, output):
        hidden = output_hidden(output)
        assert hidden.dtype == torch.bfloat16
        captured['native'] = hidden[:, -1].detach().float().cpu().numpy().copy()
    handles.append(DECODER_LAYERS[SELECTED_LAYER].register_forward_hook(legacy_hook))
    handles.append(DECODER_LAYERS[SELECTED_LAYER].register_forward_hook(native_hook))
    try:
        with torch.inference_mode():
            MODEL_FORWARD_ATTEMPTS += 1
            MODEL_EXAMPLES_ATTEMPTED += len(examples)
            result = adapter.model(**inputs, use_cache=False)
            scores = torch.log_softmax(result.logits[:, -1, :].float(), dim=-1)
            scored = [{label: float(scores[i, plan['token_by_label'][label]].cpu()) for label in ex.labels}
                      for i, ex in enumerate(examples)]
    finally:
        for handle in handles:
            handle.remove()
    assert set(captured) == {'native','fp16'}
    assert captured['native'].shape == captured['fp16'].shape == (len(examples), len(PROBE_MEAN))
    assert all(np.isfinite(x).all() for x in captured.values())
    records = []
    for ex, score in zip(examples, scored):
        label = max(ex.labels, key=score.get)
        records.append(dict(scene_id=str(ex.scene_id), variant=str(ex.variant), example_id=str(ex.example_id),
            gold=str(ex.gold_semantic), predicted=str(ex.label_to_semantic[label]), predicted_label=str(label),
            label_to_semantic={str(k): str(v) for k, v in ex.label_to_semantic.items()}, candidate_logprobs=score))
    return captured, records


def cached_bridge_capture(key, compute):
    folder = CAPTURE_CACHE/digest(key)[:24]
    folder.mkdir(parents=True, exist_ok=True)
    metadata_path, arrays_path = folder/'capture.json', folder/'states.npz'
    if metadata_path.is_file():
        metadata = read_json(metadata_path)
        assert metadata['identity'] == CAPTURE_IDENTITY and metadata['key'] == key
        assert arrays_path.is_file() and file_hash(arrays_path) == metadata['arrays_sha256']
        with np.load(arrays_path, allow_pickle=False) as source:
            arrays = {k: source[k].copy() for k in source.files}
        return metadata['result'], arrays, 0
    result, arrays, forwards = compute()
    temporary = arrays_path.with_suffix('.npz.tmp')
    with temporary.open('wb') as f:
        np.savez_compressed(f, **arrays)
    temporary.replace(arrays_path)
    save_json(metadata_path, dict(identity=CAPTURE_IDENTITY, key=key, arrays_sha256=file_hash(arrays_path), result=result))
    return result, arrays, forwards


def capture_single_bridge(example):
    old_inputs, old_plan = legacy_prepare([example])
    new_inputs, new_plan = current_prepare(example)
    old_fingerprint, new_fingerprint = tensor_signature(old_inputs), tensor_signature(new_inputs)
    differences = exact_input_comparison(old_fingerprint, new_fingerprint)
    equal = all(d['equal'] for d in differences) and old_plan == new_plan
    old_rows, new_rows = prepared_rows(old_inputs, [example]), prepared_rows(new_inputs, [example])
    assert new_rows[0]['selected_token_active'], 'Current single terminal token is padding.'
    current, current_records = execute_bridge_forward(new_inputs, [example], new_plan)
    forwards = 1
    if equal:
        legacy, legacy_records = current, current_records
    else:
        legacy, legacy_records = execute_bridge_forward(old_inputs, [example], old_plan)
        forwards += 1
    record = dict(key=list(capture_key(current_records[0])), shared_forward=equal,
        preparation_equal=equal, prepared_field_comparison=differences, legacy_plan=old_plan, current_plan=new_plan,
        legacy_prepared_rows=old_rows, current_prepared_rows=new_rows,
        legacy_record=legacy_records[0], current_record=current_records[0],
        native_to_fp16_observer_gap=float(np.max(abs(current['native'].astype('float16').astype('float32')-current['fp16']))))
    arrays = dict(current_single_native=current['native'], current_single_fp16=current['fp16'],
                  legacy_single_native=legacy['native'], legacy_single_fp16=legacy['fp16'])
    return record, arrays, forwards


def capture_batch_bridge(examples):
    inputs, plan = legacy_prepare(examples)
    rows = prepared_rows(inputs, examples)
    captures, records = execute_bridge_forward(inputs, examples, plan)
    result = dict(keys=[list(capture_key(r)) for r in records], plan=plan,
        full_inputs_fingerprint=tensor_signature(inputs), prepared_rows=rows, records=records,
        native_to_fp16_observer_gap=float(np.max(abs(captures['native'].astype('float16').astype('float32')-captures['fp16']))))
    # A padded terminal token is retained as a diagnostic finding rather than
    # silently repaired: this route reproduces the template's [:, -1] selection.
    return result, dict(legacy_batch_native=captures['native'],legacy_batch_fp16=captures['fp16']), 1


Define the two read-only capture conventions:  40%|####      | 6/15 [00:00<?, ?cell/s]

In [9]:
%%cell_progress 8/15 Load the frozen readout, exact development examples and historical provenance
try:
    from visual_torque_toolkit.behavior import BehaviorRunConfig,PromptCondition,load_behavior_examples
    READOUT_ROOT=PROJECT_ROOT/'results/cross_model_readout_correction_v2/models'/MODEL_SLUG
    DISCOVERY_ROOT=PROJECT_ROOT/'results'/MODEL_SLUG/'activation_discovery_v1'
    probe_path=packet.track(READOUT_ROOT/'frozen_probe.json')
    old_protocol=read_json(packet.track(READOUT_ROOT/'protocol.json'))
    old_runtime=read_json(packet.track(READOUT_ROOT/'runtime_identity.json'))
    capture_config=read_json(packet.track(DISCOVERY_ROOT/'capture_config.json'))
    history_identity=capture_config['identity']
    probe=read_json(probe_path)
    expected_capture_config=probe.get('fit_provenance',{}).get('files',{}).get(str(DISCOVERY_ROOT/'capture_config.json'))
    if expected_capture_config is not None:assert normalize_sha(expected_capture_config)==file_hash(DISCOVERY_ROOT/'capture_config.json')
    assert old_protocol['model_id']==MODEL_ID and old_protocol['revision']==MODEL_REVISION
    assert history_identity['model_id']==MODEL_ID and history_identity['resolved_revision']==MODEL_REVISION
    assert old_protocol['dtype']==history_identity['dtype']==MODEL_DTYPE and old_protocol['batch_size']==1
    SELECTED_LAYER=int(probe['layer']);CAPTURE_LAYERS=(SELECTED_LAYER,)
    assert SELECTED_LAYER in list(map(int,history_identity['layer_indices']))
    PROBE_MEAN=np.asarray(probe['mean'],np.float32);PROBE_SCALE=np.asarray(probe['scale'],np.float32)
    PROBE_COEF=np.asarray(probe['coef'],np.float32);PROBE_INTERCEPT=float(probe['intercept'])
    assert PROBE_MEAN.shape==PROBE_SCALE.shape==PROBE_COEF.shape and (PROBE_SCALE>0).all()
    assert all(np.isfinite(v).all() for v in (PROBE_MEAN,PROBE_SCALE,PROBE_COEF)) and np.isfinite(PROBE_INTERCEPT)
    BALANCE_THRESHOLD=float(old_protocol['balance_threshold']);READOUT_TOLERANCE=float(old_protocol['readout_tolerance'])
    manifest_path=packet.track(PROJECT_ROOT/'results/behavior_comparison_v3/activation_pilot_manifest.jsonl')
    manifest=read_jsonl(manifest_path)
    assert len(manifest)==254 and len({str(r['scene_id']) for r in manifest})==254
    assert {str(r['split']) for r in manifest}=={'probe_train','probe_dev'}
    dev_ids={str(r['scene_id']) for r in manifest if r['split']=='probe_dev'}
    assert len(dev_ids)==19 and dev_ids<=set(map(str,probe['fit_scene_ids']))
    assert normalize_sha(history_identity['activation_manifest_sha256'])==file_hash(manifest_path)
    example_config=BehaviorRunConfig(dataset_dir=DATASET_DIR,output_dir=OUTPUT_ROOT/'unused',model_id=MODEL_ID,
        model_revision=MODEL_REVISION,conditions=(PromptCondition.IMAGE_RULE,),question_kinds=('outcome',),
        splits=('probe_train','probe_dev'),batch_size=1,generation_every=0,seed=20260817)
    assert example_config.dataset_fingerprint()==history_identity['dataset_fingerprint']==old_runtime['dataset']
    all_ids={str(r['scene_id']) for r in manifest}
    all_examples=[e for e in load_behavior_examples(example_config) if str(e.scene_id) in all_ids]
    EXAMPLES={(str(e.scene_id),str(e.variant)):e for e in all_examples}
    assert len(all_examples)==len(EXAMPLES)==1524
    WANTED=sorted(k for k in EXAMPLES if k[0] in dev_ids)
    assert len(WANTED)==114 and all(sum(k[0]==sid for k in WANTED)==6 for sid in dev_ids)
    for sid in dev_ids:
        assert len({tuple(EXAMPLES[k].option_semantics) for k in WANTED if k[0]==sid})==6
    packet.json('historical_capture_configuration.json',capture_config)
    packet.json('frozen_readout.json',probe)
    packet.json('prior_E1_runtime.json',old_runtime)
    packet.json('capture_source_provenance.json',CAPTURE_SOURCE_PROVENANCE)
    logged_batch=history_identity.get('batch_size',capture_config.get('batch_size'))
    BATCH_SIZE=int(BATCH_SIZE_OVERRIDE if BATCH_SIZE_OVERRIDE is not None else logged_batch if logged_batch is not None else SPEC['template_batch'])
    assert 1<BATCH_SIZE<=8
    BATCH_SIZE_SOURCE='user_override' if BATCH_SIZE_OVERRIDE is not None else 'historical_configuration' if logged_batch is not None else 'local_template_default_not_historically_verified'
    print('冻结层：',SELECTED_LAYER,'；开发场景：19；映射：114；候选批大小：',BATCH_SIZE,'；来源：',BATCH_SIZE_SOURCE)
    print('沿用原读出器和阈值，不重新拟合。本诊断使用旧开发集，不报告新的泛化准确率。')
except Exception as error:
    bridge_issue('frozen_inputs',error)


Load the frozen readout, exact development examples and historical provenance:  47%|####6     | 7/15 [00:00<?,…

冻结层： 13 ；开发场景：19；映射：114；候选批大小： 4 ；来源： local_template_default_not_historically_verified
沿用原读出器和阈值，不重新拟合。本诊断使用旧开发集，不报告新的泛化准确率。


In [10]:
%%cell_progress 9/15 Read per-mapping archives and verify how the old scene means were formed
if ready:
    try:
        mean_path=DISCOVERY_ROOT/'scene_mean_activations.pt'
        if mean_path.is_file():
            packet.track(mean_path)
            expected=probe.get('fit_provenance',{}).get('files',{}).get(str(mean_path))
            if expected is not None:assert normalize_sha(expected)==file_hash(mean_path)
            payload=torch.load(mean_path,map_location='cpu',weights_only=True)
            pos=list(map(int,payload['layer_indices'])).index(SELECTED_LAYER)
            assert len(payload['metadata'])==len(payload['activations'])
            for i,record in enumerate(payload['metadata']):
                sid=str(record['scene_id'])
                if sid in dev_ids:
                    assert sid not in historical_means and record['split']=='probe_dev'
                    historical_means[sid]=payload['activations'][i,pos].float().numpy().copy()
            assert set(historical_means)==dev_ids
            assert all(np.isfinite(x).all() and x.shape==PROBE_MEAN.shape for x in historical_means.values())
            del payload
        else:
            coverage.append(dict(stage='historical_scene_means',status='missing',reason=str(mean_path)))
        shard_paths=sorted(DISCOVERY_ROOT.glob('shard_*.pt'))
        for path in tqdm(shard_paths,desc='Reading archived per-mapping states'):
            payload=torch.load(packet.track(path),map_location='cpu',weights_only=True)
            pos=list(map(int,payload['layer_indices'])).index(SELECTED_LAYER)
            assert len(payload['records'])==len(payload['activations'])
            for i,record in enumerate(payload['records']):
                key=capture_key(record)
                assert key in EXAMPLES and key not in historical_records
                ex=EXAMPLES[key]
                assert str(record['example_id'])==str(ex.example_id)
                assert str(record['gold_semantic'])==str(ex.gold_semantic) and int(record['net_torque'])==int(ex.net_torque)
                mapping={str(k):str(v) for k,v in record['label_to_semantic'].items()}
                assert mapping=={str(k):str(v) for k,v in ex.label_to_semantic.items()}
                archive_order.append(key);historical_records[key]=record
                if key in WANTED:
                    historical_states[key]=payload['activations'][i,pos].float().numpy().copy()
            del payload
        if shard_paths:
            assert set(historical_records)==set(EXAMPLES) and len(historical_states)==114,'Incomplete historical shard collection'
            assert all(np.isfinite(x).all() and x.shape==PROBE_MEAN.shape for x in historical_states.values())
            for sid in sorted(dev_ids):
                # Match the original shard-record summation order exactly.
                keys=[k for k in archive_order if k[0]==sid]
                rebuilt=np.stack([historical_states[k] for k in keys]).mean(0).astype('float16').astype('float32')
                if sid in historical_means:
                    difference=float(np.max(abs(rebuilt-historical_means[sid])))
                    archive_mean_checks.append(dict(scene_id=sid,hidden_max_difference=difference,
                        readout_difference=decode(rebuilt)-decode(historical_means[sid]),exact_match=difference==0))
            source_original_mean_consistent=all(r['exact_match'] for r in archive_mean_checks) if len(archive_mean_checks)==19 else None
            schedule_source='saved_shard_record_order; batch boundaries inferred, not historically logged'
        else:
            archive_order=[(str(e.scene_id),str(e.variant)) for e in all_examples]
            schedule_source='current reconstruction of original full manifest order; no historical per-mapping archive'
            coverage.append(dict(stage='historical_variant_shards',status='missing',reason='No shard_*.pt; continue with explicitly unverified candidate batch context'))
        WANTED=[k for k in archive_order if k[0] in dev_ids]
        assert len(WANTED)==len(set(WANTED))==114
        BATCHES=candidate_batch_schedule(archive_order,set(WANTED),BATCH_SIZE)
        USED=sorted({k for batch in BATCHES for k in batch}|set(WANTED))
        stamps={}
        for key in USED:
            ex=EXAMPLES[key];packet.track(ex.image_path)
            stamps[ex.example_id]=dict(scene_id=key[0],variant=key[1],split=ex.split,prompt=ex.prompt,
                image_sha256=file_hash(ex.image_path),labels=list(ex.labels),semantics=list(ex.option_semantics),gold=str(ex.gold_semantic))
        for path in sorted((TOOLKIT_ROOT/'src/visual_torque_toolkit').rglob('*.py')):packet.track(path)
        packet.json('example_inputs.json',stamps)
        packet.json('batch_schedule.json',dict(batch_size=BATCH_SIZE,batch_size_source=BATCH_SIZE_SOURCE,
            order_source=schedule_source,original_batch_membership_verified=False,
            mean_aggregation_order='within-scene record order from the stated schedule source; shared by all compared routes',
            batches=[[list(k) for k in b] for b in BATCHES],evaluated_development_mappings=114,
            historical_training_companions=len([k for k in USED if k not in WANTED])))
        packet.csv('historical_mean_rebuild.csv',pd.DataFrame(archive_mean_checks))
        diagnostic_show('历史逐映射分片能否重建原均值',pd.DataFrame(archive_mean_checks),19)
        print('批次比较只统计19个开发场景；用于恢复候选批次上下文的旧训练图片数量：',len(USED)-len(WANTED))
        print('单张最多228次前向（输入相同则114次），另加候选批次：',len(BATCHES),'；不做重复性重跑。')
    except Exception as error:
        bridge_issue('historical_archives',error)


Read per-mapping archives and verify how the old scene means were formed:  53%|#####3    | 8/15 [00:00<?, ?cel…

Reading archived per-mapping states:   0%|          | 0/24 [00:00<?, ?it/s]


历史逐映射分片能否重建原均值


,scene_id,hidden_max_difference,readout_difference,exact_match
0,multi_03b698958cd84596,0.0,0.0,True
1,multi_c0e457437617298d,0.0,0.0,True
2,multi_d6b66fc6f2ec3bae,0.0,0.0,True
3,multi_e596431a55f8cd96,0.0,0.0,True
4,multi_f6b194290933e01e,0.0,0.0,True
5,single_lu1_ld2_ru2_rd4,0.0,0.0,True
6,single_lu1_ld3_ru3_rd1,0.0,0.0,True
7,single_lu1_ld4_ru2_rd5,0.0,0.0,True
8,single_lu1_ld4_ru3_rd1,0.0,0.0,True
9,single_lu1_ld4_ru4_rd2,0.0,0.0,True


批次比较只统计19个开发场景；用于恢复候选批次上下文的旧训练图片数量： 30
单张最多228次前向（输入相同则114次），另加候选批次： 36 ；不做重复性重跑。


In [11]:
%%cell_progress 10/15 Load the same pinned model for all paired routes
if ready:
    try:
        if 'DECODER_LAYERS' in globals():
            del DECODER_LAYERS
        cell_stage(f'Loading {MODEL_KEY} weights for the capture comparison')
        import gc
        if 'adapter' in globals():
            del adapter
            gc.collect()
            torch.cuda.empty_cache()

        adapter = HuggingFaceMultimodalAdapter(
            MODEL_ID, revision=MODEL_REVISION,
            dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
        )
        display(adapter.metadata)
        assert adapter.metadata['resolved_revision'] == MODEL_REVISION

        def resolve_module_path(root, path):
            value = root
            for component in path.split('.'):
                value = getattr(value, component)
            return value

        def discover_decoder_layers(model):
            candidates = (
                'model.language_model.layers', 'model.text_model.layers',
                'language_model.model.layers', 'model.model.layers',
                'model.layers', 'language_model.layers',
            )
            for path in candidates:
                try:
                    layers = resolve_module_path(model, path)
                except AttributeError:
                    continue
                if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
                    return path, layers
            raise RuntimeError('Could not locate decoder layers.')

        LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
        assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
        assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
        print('Layer path:', LAYER_PATH)
        print('Capture layers:', CAPTURE_LAYERS)
        assert not any(v == 'disk' for v in getattr(adapter.model,'hf_device_map',{}).values()), 'Use a GPU with enough memory for this model.'
    except Exception as error:
        bridge_issue('model_load',error)


Load the same pinned model for all paired routes:  60%|######    | 9/15 [00:00<?, ?cell/s]

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

{'adapter': 'HuggingFaceMultimodalAdapter',
 'model_id': 'Qwen/Qwen3.5-9B',
 'requested_revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a',
 'resolved_revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a',
 'dtype': 'bfloat16',
 'device_map': 'auto',
 'parameter_count': 9409813744,
 'torch_version': '2.11.0+cu128',
 'transformers_version': '5.14.0'}

Layer path: model.language_model.layers
Capture layers: (13,)


In [12]:
%%cell_progress 11/15 Freeze this current runtime and an isolated resumable capture cache
if ready:
    try:
        import transformers
        adapter.model.eval()
        assert adapter.metadata['resolved_revision']==MODEL_REVISION
        assert LAYER_PATH==history_identity['layer_path']
        runtime=dict(torch=torch.__version__,transformers=transformers.__version__,cuda=torch.version.cuda,
            gpu=torch.cuda.get_device_name(0),cuda_capability=list(torch.cuda.get_device_capability(0)),
            dtype=MODEL_DTYPE,layer_path=LAYER_PATH,checkpoint=MODEL_REVISION,
            attention_implementation=str(getattr(adapter.model.config,'_attn_implementation',None)),
            tf32_matmul=bool(torch.backends.cuda.matmul.allow_tf32),cudnn_benchmark=bool(torch.backends.cudnn.benchmark),
            adapter_metadata=adapter.metadata,
            toolkit={str(p.relative_to(TOOLKIT_ROOT)):file_hash(p) for p in sorted((TOOLKIT_ROOT/'src/visual_torque_toolkit').rglob('*.py'))})
        CAPTURE_IDENTITY=json.loads(json.dumps(dict(schema='capture-bridge-cache-1.0',implementation=IMPLEMENTATION_SHA256,
            runtime=runtime,inputs=dict(packet.inputs),schedule=[[list(k) for k in b] for b in BATCHES],
            probe=file_hash(probe_path),layer=SELECTED_LAYER,batch_size=BATCH_SIZE,batch_size_source=BATCH_SIZE_SOURCE),default=plain))
        CAPTURE_CACHE=OUTPUT_ROOT/'captures'/digest(CAPTURE_IDENTITY)[:20]
        CAPTURE_CACHE.mkdir(parents=True,exist_ok=True)
        freeze(CAPTURE_CACHE/'identity.json',CAPTURE_IDENTITY)
        packet.json('current_runtime.json',runtime)
        packet.json('capture_identity.json',CAPTURE_IDENTITY)
        runtime_rows=[dict(field=k,prior_E1=json.dumps(old_runtime.get(k),default=plain),current=json.dumps(runtime.get(k),default=plain),
            equal=old_runtime.get(k)==runtime.get(k)) for k in ('torch','transformers','cuda','dtype','checkpoint','layer_path')]
        packet.csv('runtime_comparison.csv',pd.DataFrame(runtime_rows))
        diagnostic_show('当前运行与原 E1 配置；未记录字段不视为一致',pd.DataFrame(runtime_rows))
        print('本运行独立缓存：',CAPTURE_CACHE)
    except Exception as error:
        bridge_issue('runtime_identity',error)


Freeze this current runtime and an isolated resumable capture cache:  67%|######6   | 10/15 [00:00<?, ?cell/s]


当前运行与原 E1 配置；未记录字段不视为一致


,field,prior_E1,current,equal
0,torch,"""2.11.0+cu128""","""2.11.0+cu128""",True
1,transformers,"""5.14.0""","""5.14.0""",True
2,cuda,null,"""12.8""",False
3,dtype,null,"""bfloat16""",False
4,checkpoint,null,"""c202236235762e1c871ad0ccb60c8ee5ba337b9a""",False
5,layer_path,null,"""model.language_model.layers""",False


本运行独立缓存： /content/drive/MyDrive/VisualTorque/results/cross_model_independent_confirmation/capture_bridge/input_mapping_fix/Qwen__Qwen3.5-9B/captures/2f75ce1d175b5660f880


In [13]:
%%cell_progress 12/15 Compare both single-image preparation and capture conventions
if ready:
    try:
        for key in tqdm(WANTED,desc='Single-image capture bridge',unit='mapping'):
            result,arrays,_=cached_bridge_capture(['single',list(key)],lambda key=key:capture_single_bridge(EXAMPLES[key]))
            assert tuple(result['key'])==key
            single_results[key]=result;single_arrays[key]=arrays
        coverage.append(dict(stage='single_capture',status='complete',mappings=len(single_results)))
        packet.json('single_capture_details.json',list(single_results.values()))
        prep_rows=[dict(scene_id=k[0],variant=k[1],preparation_equal=r['preparation_equal'],shared_forward=r['shared_forward'],
            fp16_observer_gap=r['native_to_fp16_observer_gap']) for k,r in single_results.items()]
        packet.csv('single_capture_preparation.csv',pd.DataFrame(prep_rows))
        diagnostic_show('单张输入是否一致，以及同次前向的捕获对照',pd.DataFrame(prep_rows),20)
    except Exception as error:
        bridge_issue('single_capture',error)


Compare both single-image preparation and capture conventions:  73%|#######3  | 11/15 [00:00<?, ?cell/s]

Single-image capture bridge:   0%|          | 0/114 [00:00<?, ?mapping/s]


单张输入是否一致，以及同次前向的捕获对照


,scene_id,variant,preparation_equal,shared_forward,fp16_observer_gap
0,multi_03b698958cd84596,permutation_0,True,True,0.0
1,multi_03b698958cd84596,permutation_1,True,True,0.0
2,multi_03b698958cd84596,permutation_2,True,True,0.0
3,multi_03b698958cd84596,permutation_3,True,True,0.0
4,multi_03b698958cd84596,permutation_4,True,True,0.0
5,multi_03b698958cd84596,permutation_5,True,True,0.0
6,multi_c0e457437617298d,permutation_0,True,True,0.0
7,multi_c0e457437617298d,permutation_1,True,True,0.0
8,multi_c0e457437617298d,permutation_2,True,True,0.0
9,multi_c0e457437617298d,permutation_3,True,True,0.0


这里显示前 20 行；结果 ZIP 保存全部 114 行。


In [14]:
%%cell_progress 13/15 Replay candidate batch contexts without changing positions, precision or inputs
if ready:
    try:
        for keys in tqdm(BATCHES,desc='Candidate historical batch contexts',unit='batch'):
            result,arrays,_=cached_bridge_capture(['candidate_batch',[list(k) for k in keys]],
                lambda keys=keys:capture_batch_bridge([EXAMPLES[k] for k in keys]))
            assert result['keys']==[list(k) for k in keys]
            batch_results.append(result)
            for i,key in enumerate(keys):
                if key in WANTED:
                    assert key not in batch_arrays
                    batch_arrays[key]={name:value[i:i+1].copy() for name,value in arrays.items()}
        assert set(batch_arrays)==set(WANTED)
        coverage.append(dict(stage='batch_capture',status='complete',batches=len(BATCHES),mappings=len(batch_arrays)))
        packet.json('batch_capture_details.json',batch_results)
    except Exception as error:
        bridge_issue('batch_capture',error)


Replay candidate batch contexts without changing positions, precision or inputs:  80%|########  | 12/15 [00:00…

Candidate historical batch contexts:   0%|          | 0/36 [00:00<?, ?batch/s]

In [15]:
%%cell_progress 14/15 Compare native states, historical variants and all nineteen scene means
scene_table=pd.DataFrame();gap_summary=pd.DataFrame();variant_table=pd.DataFrame();notes=[]
if ready:
    try:
        routes={name:np.concatenate([single_arrays[k][name] for k in WANTED],axis=0) for name in next(iter(single_arrays.values()))}
        routes.update({name:np.concatenate([batch_arrays[k][name] for k in WANTED],axis=0) for name in next(iter(batch_arrays.values()))})
        if set(historical_states)==set(WANTED):routes['historical_variants_fp16']=np.stack([historical_states[k] for k in WANTED])
        batch_by_key={tuple(key):(result,i) for result in batch_results for i,key in enumerate(result['keys']) if tuple(key) in WANTED}
        variant_rows=[]
        for i,key in enumerate(WANTED):
            single=single_results[key];batch,bi=batch_by_key[key]
            a=single['legacy_prepared_rows'][0];b=batch['prepared_rows'][bi]
            scores_a=single['legacy_record']['candidate_logprobs'];scores_b=batch['records'][bi]['candidate_logprobs']
            x=np.array([scores_a[k] for k in ('A','B','C')]);y=np.array([scores_b[k] for k in ('A','B','C')])
            row=dict(scene_id=key[0],variant=key[1],single_preparation_equal=single['preparation_equal'],
                single_terminal_active=a['selected_token_active'],batch_terminal_active=b['selected_token_active'],
                unpadded_tokens_equal=a['unpadded_input_ids']==b['unpadded_input_ids'],
                image_partition=a['image_partition']+' / '+b['image_partition'],
                image_fingerprint_equal=None if a['image_fingerprint'] is None or b['image_fingerprint'] is None else a['image_fingerprint']==b['image_fingerprint'],
                batch_vs_single_hidden_max=float(np.max(abs(routes['legacy_batch_native'][i]-routes['legacy_single_native'][i]))),
                batch_vs_single_readout=decode(routes['legacy_batch_native'][i])-decode(routes['legacy_single_native'][i]),
                batch_vs_single_centered_score_max=float(np.max(abs((y-y.mean())-(x-x.mean())))),
                batch_vs_single_answer_changed=single['legacy_record']['predicted']!=batch['records'][bi]['predicted'])
            for name,array in routes.items():
                row[name+'_readout']=decode(array[i])
                if key in historical_states:
                    row[name+'_historical_hidden_max']=float(np.max(abs(array[i]-historical_states[key])))
                    row[name+'_historical_readout_gap']=decode(array[i])-decode(historical_states[key])
            variant_rows.append(row)
        variant_table=pd.DataFrame(variant_rows)
        scene_table=aggregate_capture_scenes(WANTED,routes,list(single_results.values()),historical_means,BALANCE_THRESHOLD,decode)
        gap_summary=scene_gap_summary(scene_table,list(routes))
        packet.csv('all_114_mapping_comparisons.csv',variant_table)
        packet.csv('all_19_scene_comparisons.csv',scene_table)
        packet.csv('historical_gap_by_route.csv',gap_summary)
        path=OUTPUT_ROOT/'selected_capture_states.npz'
        with path.open('wb') as f:np.savez_compressed(f,**routes,keys=np.asarray(WANTED),
            historical_scene_ids=np.asarray(sorted(historical_means)),
            historical_scene_means=np.stack([historical_means[s] for s in sorted(historical_means)]) if historical_means else np.empty((0,len(PROBE_MEAN))))
        packet.outputs.add(path)
        repeat_path=PROJECT_ROOT/'results/cross_model_independent_confirmation/readout_repeatability'/REPEAT_RUN_ID/MODEL_SLUG/'analysis/historical_scene_mean_comparison.csv'
        if repeat_path.is_file():
            previous=pd.read_csv(packet.track(repeat_path))
            prior=previous.groupby('scene_id').current_native_readout.mean().rename('prior_repeat_current_readout')
            joined=scene_table[['scene_id','current_single_native_readout']].merge(prior,on='scene_id',validate='one_to_one')
            assert len(joined)==19
            joined['current_minus_prior_repeat']=joined.current_single_native_readout-joined.prior_repeat_current_readout
            packet.csv('current_vs_previous_repeatability_run.csv',joined)
        else:
            coverage.append(dict(stage='previous_repeat_comparison',status='missing',reason=str(repeat_path)))
        storage_ok=all(r['native_to_fp16_observer_gap']==0 for r in single_results.values()) and all(r['native_to_fp16_observer_gap']==0 for r in batch_results)
        notes=bridge_classification(variant_table.single_preparation_equal.all(),storage_ok,source_original_mean_consistent,
            gap_summary,float(variant_table.batch_vs_single_readout.abs().max()))
        diagnostic_show('同一19场景：哪种捕获方式更接近历史读出',gap_summary,10)
        diagnostic_show('完整19场景的读出与分类',scene_table,19)
        diagnostic_show('批处理的文本、图像和位置是否一致',variant_table[['scene_id','variant','unpadded_tokens_equal','image_fingerprint_equal','batch_terminal_active','batch_vs_single_readout','batch_vs_single_answer_changed']],20)
        for note in notes:print('•',note)
    except Exception as error:
        bridge_issue('comparison_analysis',error)


Compare native states, historical variants and all nineteen scene means:  87%|########6 | 13/15 [00:00<?, ?cel…


同一19场景：哪种捕获方式更接近历史读出


,route,scenes,mean_absolute_archive_gap,maximum_absolute_archive_gap,mean_signed_archive_gap,class_changed_scenes
0,current_single_native,19,1.399750,4.786862,1.323115,6
1,current_single_fp16,19,1.403767,4.794667,1.323032,6
2,legacy_single_native,19,1.399750,4.786862,1.323115,6
3,legacy_single_fp16,19,1.403767,4.794667,1.323032,6
4,legacy_batch_native,19,1.292943,3.771564,1.009326,8
5,legacy_batch_fp16,19,1.294309,3.786697,1.015496,8
6,historical_variants_fp16,19,0.000000,0.000000,0.000000,0



完整19场景的读出与分类


,scene_id,mappings,historical_readout,historical_class,current_single_native_readout,current_single_native_class,current_single_native_archive_gap,current_single_native_archive_class_changed,current_single_fp16_readout,current_single_fp16_class,current_single_fp16_archive_gap,current_single_fp16_archive_class_changed,legacy_single_native_readout,legacy_single_native_class,legacy_single_native_archive_gap,legacy_single_native_archive_class_changed,legacy_single_fp16_readout,legacy_single_fp16_class,legacy_single_fp16_archive_gap,legacy_single_fp16_archive_class_changed,legacy_batch_native_readout,legacy_batch_native_class,legacy_batch_native_archive_gap,legacy_batch_native_archive_class_changed,legacy_batch_fp16_readout,legacy_batch_fp16_class,legacy_batch_fp16_archive_gap,legacy_batch_fp16_archive_class_changed,historical_variants_fp16_readout,historical_variants_fp16_class,historical_variants_fp16_archive_gap,historical_variants_fp16_archive_class_changed
0,multi_03b698958cd84596,6,-28.014347,right,-25.382525,right,2.631822,False,-25.365719,right,2.648628,False,-25.382525,right,2.631822,False,-25.365719,right,2.648628,False,-24.462221,right,3.552126,False,-24.466579,right,3.547768,False,-28.014347,right,0.0,False
1,multi_c0e457437617298d,6,-2.020667,right,-1.794490,right,0.226177,False,-1.793668,right,0.226999,False,-1.794490,right,0.226177,False,-1.793668,right,0.226999,False,-0.030597,balanced,1.990070,True,-0.033083,balanced,1.987584,True,-2.020667,right,0.0,False
2,multi_d6b66fc6f2ec3bae,6,-1.997707,right,1.089819,left,3.087526,True,1.093174,left,3.090882,True,1.089819,left,3.087526,True,1.093174,left,3.090882,True,-0.410842,balanced,1.586865,True,-0.425151,balanced,1.572556,True,-1.997707,right,0.0,False
3,multi_e596431a55f8cd96,6,2.981841,left,4.028483,left,1.046642,False,4.009651,left,1.027810,False,4.028483,left,1.046642,False,4.009651,left,1.027810,False,5.057807,left,2.075966,False,5.058588,left,2.076747,False,2.981841,left,0.0,False
4,multi_f6b194290933e01e,6,-8.989764,right,-6.675695,right,2.314069,False,-6.724561,right,2.265203,False,-6.675695,right,2.314069,False,-6.724561,right,2.265203,False,-9.261181,right,-0.271417,False,-9.244983,right,-0.255219,False,-8.989764,right,0.0,False
5,single_lu1_ld2_ru2_rd4,6,-5.998769,right,-6.114920,right,-0.116151,False,-6.107886,right,-0.109117,False,-6.114920,right,-0.116151,False,-6.107886,right,-0.109117,False,-6.919001,right,-0.920232,False,-6.895965,right,-0.897196,False,-5.998769,right,0.0,False
6,single_lu1_ld3_ru3_rd1,6,0.005899,balanced,0.704335,left,0.698436,True,0.715207,left,0.709308,True,0.704335,left,0.698436,True,0.715207,left,0.709308,True,-0.842078,right,-0.847977,True,-0.847674,right,-0.853574,True,0.005899,balanced,0.0,False
7,single_lu1_ld4_ru2_rd5,6,-5.983981,right,-6.208163,right,-0.224182,False,-6.256579,right,-0.272598,False,-6.208163,right,-0.224182,False,-6.256579,right,-0.272598,False,-4.958860,right,1.025121,False,-4.930330,right,1.053651,False,-5.983981,right,0.0,False
8,single_lu1_ld4_ru3_rd1,6,0.963423,left,0.582753,left,-0.380670,False,0.585931,left,-0.377492,False,0.582753,left,-0.380670,False,0.585931,left,-0.377492,False,1.583649,left,0.620227,False,1.597573,left,0.634150,False,0.963423,left,0.0,False
9,single_lu1_ld4_ru4_rd2,6,-3.992990,right,-2.903949,right,1.089041,False,-2.911267,right,1.081723,False,-2.903949,right,1.089041,False,-2.911267,right,1.081723,False,-3.667142,right,0.325848,False,-3.704634,right,0.288356,False,-3.992990,right,0.0,False



批处理的文本、图像和位置是否一致


,scene_id,variant,unpadded_tokens_equal,image_fingerprint_equal,batch_terminal_active,batch_vs_single_readout,batch_vs_single_answer_changed
0,multi_03b698958cd84596,permutation_0,True,True,True,1.223188,False
1,multi_03b698958cd84596,permutation_1,True,True,True,-2.959081,False
2,multi_03b698958cd84596,permutation_2,True,True,True,2.595874,False
3,multi_03b698958cd84596,permutation_3,True,True,True,-0.898304,False
4,multi_03b698958cd84596,permutation_4,True,True,True,2.146020,False
5,multi_03b698958cd84596,permutation_5,True,True,True,3.414114,False
6,multi_c0e457437617298d,permutation_0,True,True,True,-1.519326,False
7,multi_c0e457437617298d,permutation_1,True,True,True,-5.789317,False
8,multi_c0e457437617298d,permutation_2,True,True,True,4.007311,False
9,multi_c0e457437617298d,permutation_3,True,True,True,-0.079242,False


这里显示前 20 行；结果 ZIP 保存全部 114 行。
• 同一当前环境中，所选批处理上下文与单张处理的差异已测量；它同时包含批处理、填充及可能的图像处理差异。
• 批处理路线与历史均值更接近。
• 较接近历史缓存不等于已恢复历史流程；未记录的批次组成、软件或硬件仍不能自动归因。此诊断不更新任何科学确认状态。


In [16]:
%%cell_progress 15/15 Download a complete or explicitly partial diagnostic packet
packet.json('single_capture_details.json',list(single_results.values()))
packet.json('batch_capture_details.json',batch_results)
packet.csv('coverage.csv',pd.DataFrame(coverage+issues))
complete=bool(ready and len(scene_table)==19 and len(single_results)==114 and len(batch_arrays)==114)
summary=dict(status=('complete_with_missing_inputs' if any(r.get('status')=='missing' for r in coverage) else 'complete') if complete else 'partial_diagnostic_read_issues',model=MODEL_KEY,
    diagnostic_workflow_complete=complete,compatibility_conclusion='diagnostic_only_no_automatic_historical_equivalence',
    development_scenes=19,single_mappings_completed=len(single_results),batch_mappings_completed=len(batch_arrays),
    new_forward_attempts_this_execution=MODEL_FORWARD_ATTEMPTS,new_examples_attempted_this_execution=MODEL_EXAMPLES_ATTEMPTED,
    original_mean_rebuild_consistent=source_original_mean_consistent,
    historical_batch_membership_verified=False,batch_size_source=globals().get('BATCH_SIZE_SOURCE','unavailable'),
    batch_size=globals().get('BATCH_SIZE'),old_scientific_decisions_unchanged=True,probe_refit=False,
    issues=issues,coverage=coverage,interpretation=notes)
report='\n\n'.join(['# 新旧捕获与批处理对照',
    '这是在同一当前环境重放旧模板逻辑的诊断，不是新科学确认，也不声称完整恢复了历史环境。',
    '## 覆盖情况',csv_markdown(pd.DataFrame(coverage+issues)),
    '## 各路线与历史均值的距离',csv_markdown(gap_summary),
    '## 19个开发场景',csv_markdown(scene_table),
    '## 解读',*notes,
    '## 使用边界',
    '模型、冻结读出器、层位和分类阈值不变。全部模型前向为 BF16；原生状态存为精确 float32 副本，另保留旧式 FP16 捕获与均值存储。',
    '旧配置没有批大小时按本地旧模板设定候选批大小（Gemma=2，Qwen-9B=4），明确标记为推定。分片顺序不包含历史批次边界或恢复运行日志，因此批次成员没有被宣称为历史已验证。',
    '六种映射的求平均顺序沿用历史分片记录顺序（分片缺失时采用当前重建顺序），全部比较路线共用该顺序。与上一份重复性运行的差别还可能包含求和顺序的影响。',
    '单张两套准备流程比较完整张量内容；批次逐图比较未填充文本及可识别的图像分块。不支持逐图拆分的图像字段标为未知，不能把所有批次差异直接归因于数值内核。',
    '批处理路线保留旧模板的最后位置读取，包括可能出现的末尾填充；不在诊断中偷偷修正输入。若末尾是填充，这就是应报告的问题。',
    '历史训练图片只用作候选批次的上下文，不参与拟合或新的科学效果统计。开发集曾参与原读出器拟合，不能报告成新的测试集泛化成绩。',
    '若当前运行与上一份重复性诊断的环境不同，跨运行差异不能叫作同配置随机性。缓存按完整运行身份隔离，不复用不同输入、软件、模型或硬件的状态。',
    '本包包含逐映射捕获状态、输入指纹、候选批次组成、原冻结读出器和完整表格。未完成的部分列入 issues；不要把 partial 状态作为问题解决。'])+'\n'
packet.finish(summary,report,'VisualTorque_Capture_Bridge_'+MODEL_KEY+'_Output.zip',AUTO_DOWNLOAD)


Download a complete or explicitly partial diagnostic packet:  93%|#########3| 14/15 [00:00<?, ?cell/s]

{
  "status": "complete",
  "model": "qwen9b",
  "diagnostic_workflow_complete": true,
  "compatibility_conclusion": "diagnostic_only_no_automatic_historical_equivalence",
  "development_scenes": 19,
  "single_mappings_completed": 114,
  "batch_mappings_completed": 114,
  "new_forward_attempts_this_execution": 150,
  "new_examples_attempted_this_execution": 258,
  "original_mean_rebuild_consistent": true,
  "historical_batch_membership_verified": false,
  "batch_size_source": "local_template_default_not_historically_verified",
  "batch_size": 4,
  "old_scientific_decisions_unchanged": true,
  "probe_refit": false,
  "issues": [],
  "coverage": [
    {
      "stage": "single_capture",
      "status": "complete",
      "mappings": 114
    },
    {
      "stage": "batch_capture",
      "status": "complete",
      "batches": 36,
      "mappings": 114
    }
  ],
  "interpretation": [
    "同一当前环境中，所选批处理上下文与单张处理的差异已测量；它同时包含批处理、填充及可能的图像处理差异。",
    "批处理路线与历史均值更接近。",
    "较接近历史缓存不等于已恢复历史流程；未记录的批

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PosixPath('/content/drive/MyDrive/VisualTorque/results/cross_model_independent_confirmation/capture_bridge/input_mapping_fix/Qwen__Qwen3.5-9B/VisualTorque_Capture_Bridge_qwen9b_Output.zip')

PosixPath('/content/drive/MyDrive/VisualTorque/results/cross_model_independent_confirmation/capture_bridge/input_mapping_fix/Qwen__Qwen3.5-9B/VisualTorque_Capture_Bridge_qwen9b_Output.zip')